<a href="https://colab.research.google.com/github/detektor777/colab_list_video/blob/main/%D0%B0uto_scene_split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ##**Install** { display-mode: "form" }

!nvidia-smi

!pip install -q scenedetect[opencv] tqdm
!apt-get -y -qq install ffmpeg > /dev/null

import scenedetect
print("PySceneDetect version:", scenedetect.__version__)
!ffmpeg -version | head -n 1


In [ ]:
#@title ##**Select Video File** { display-mode: "form" }
import os
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
from google.colab import drive

upload_option = "Load from Google Drive Root"  #@param ["Upload from PC", "Load from Google Drive Root", "Load from Google Drive"]

file_name = None
last_selected_button = None

def reset_button_colors(buttons):
    for btn in buttons:
        btn.style.button_color = None

if upload_option == "Upload from PC":
    print("Please upload a video file.")
    uploaded = files.upload()
    if uploaded:
        file_name = list(uploaded.keys())[0]
    else:
        print("No file uploaded.")
        file_name = None

elif upload_option == "Load from Google Drive Root":
    drive.mount('/content/drive')
    root_dir = '/content/drive/MyDrive/'

    video_extensions = ['.mp4', '.mkv', '.avi', '.mov']
    files_list = []

    for f in os.listdir(root_dir):
        if os.path.isfile(os.path.join(root_dir, f)) and os.path.splitext(f)[1].lower() in video_extensions:
            files_list.append(f)

    if not files_list:
        print("No video files found in Google Drive root.")
        file_name = None
    else:
        print("Select a video file from Google Drive root:")

        output = widgets.Output()
        buttons = []

        def on_button_clicked(b):
            global file_name, last_selected_button
            with output:
                clear_output()
                reset_button_colors(buttons)
                selected_file = b.description
                file_name = os.path.join(root_dir, selected_file)

                if file_name and os.path.exists(file_name):
                    b.style.button_color = 'green'
                else:
                    b.style.button_color = 'red'

                last_selected_button = b
                print(f"Selected file: {file_name if file_name else 'None'}")

        for file in files_list:
            button = widgets.Button(description=file, layout=widgets.Layout(width='500px', overflow='hidden', text_overflow='ellipsis'))
            button.on_click(on_button_clicked)
            buttons.append(button)

        display(widgets.VBox(buttons), output)

elif upload_option == "Load from Google Drive":
    drive.mount('/content/drive')
    root_dir = '/content/drive/MyDrive/'

    video_extensions = ['.mp4', '.mkv', '.avi', '.mov']
    files_list = []

    for dirpath, _, filenames in os.walk(root_dir):
        for f in filenames:
            if os.path.splitext(f)[1].lower() in video_extensions:
                relative_path = os.path.relpath(os.path.join(dirpath, f), root_dir)
                files_list.append(relative_path)

    if not files_list:
        print("No video files found in Google Drive or its subfolders.")
        file_name = None
    else:
        print("Select a video file from Google Drive (including subfolders):")

        output = widgets.Output()
        buttons = []

        def on_button_clicked(b):
            global file_name, last_selected_button
            with output:
                clear_output()
                reset_button_colors(buttons)
                selected_file = b.description
                file_name = os.path.join(root_dir, selected_file)

                if file_name and os.path.exists(file_name):
                    b.style.button_color = 'green'
                else:
                    b.style.button_color = 'red'

                last_selected_button = b
                print(f"Selected file: {file_name if file_name else 'None'}")

        for file in files_list:
            button = widgets.Button(description=file, layout=widgets.Layout(width='500px', overflow='hidden', text_overflow='ellipsis'))
            button.on_click(on_button_clicked)
            buttons.append(button)

        display(widgets.VBox(buttons), output)

if file_name:
    print(f"Video file path set to: {file_name}")
else:
    print("Video file path not set. Please select a file.")


In [ ]:
#@title ##**Configuration** { display-mode: "form" }

# Детектор сцен:
#  - ContentDetector  — базовий, реагує на різку зміну кадру (найуніверсальніший)
#  - AdaptiveDetector — те саме, але з адаптивним порогом (краще для панорам/руху камери)
#  - ThresholdDetector — реагує на затемнення/освітлення (fade in/out), для монтажних склеек
detector_type = "ContentDetector"  #@param ["ContentDetector", "AdaptiveDetector", "ThresholdDetector"]

# Поріг чутливості: менше значення = більше сцен знайдено
threshold = 27  #@param {type:"slider", min:1, max:100, step:1}

# Мінімальна довжина сцени в секундах (щоб не різало на дуже дрібні шматки)
min_scene_len_sec = 0.6  #@param {type:"slider", min:0.1, max:5, step:0.1}

# Куди зберегти архів зі сценами
output_folder = "google_drive"  #@param ["google_drive", "content"]

print(f"Video file:        {file_name}")
print(f"Detector:          {detector_type}")
print(f"Threshold:         {threshold}")
print(f"Min scene length:  {min_scene_len_sec} s")
print(f"Output folder:     {output_folder}")


In [ ]:
#@title ##**Detect Scenes & Merge (optional)** { display-mode: "form" }
import os
import ipywidgets as widgets
from IPython.display import display, clear_output
from scenedetect import open_video, SceneManager
from scenedetect.detectors import ContentDetector, AdaptiveDetector, ThresholdDetector

if not file_name or not os.path.exists(file_name):
    raise FileNotFoundError("Video file not found. Run the 'Select Video File' cell first.")

# merge_ranges створюємо одразу, щоб змінна існувала навіть якщо детекція нижче впаде з помилкою
merge_ranges = []

video = open_video(file_name)
min_scene_len_frames = max(1, int(min_scene_len_sec * video.frame_rate))

scene_manager = SceneManager()
if detector_type == "ContentDetector":
    scene_manager.add_detector(ContentDetector(threshold=threshold, min_scene_len=min_scene_len_frames))
elif detector_type == "AdaptiveDetector":
    # У AdaptiveDetector зовсім інша шкала порогу (типове значення ~3.0, а не ~27,
    # як у ContentDetector), тому переводимо значення повзунка в цю шкалу.
    adaptive_threshold = max(0.5, threshold / 10)
    scene_manager.add_detector(AdaptiveDetector(adaptive_threshold=adaptive_threshold, min_scene_len=min_scene_len_frames))
else:
    scene_manager.add_detector(ThresholdDetector(threshold=threshold, min_scene_len=min_scene_len_frames))

print("Detecting scenes...")
scene_manager.detect_scenes(video=video, show_progress=True)
scene_list = scene_manager.get_scene_list()

if not scene_list:
    raise RuntimeError(
        "No scenes detected. Спробуй зменшити threshold (для AdaptiveDetector корисний "
        "діапазон повзунка приблизно 15-40, що відповідає adaptive_threshold ~1.5-4.0)."
    )

print(f"\nDetected {len(scene_list)} scenes:\n")
for i, (start, end) in enumerate(scene_list, start=1):
    duration = end.get_seconds() - start.get_seconds()
    print(f"  Scene {i:>3}: {start.get_timecode()} -> {end.get_timecode()}  ({duration:.2f} s)")

# --- Ручне об'єднання сцен повзунком, якщо детектор помилково розрізав одну ---
# --- сцену на дві (або більше) -----------------------------------------------
range_slider = widgets.IntRangeSlider(
    value=[1, min(2, len(scene_list))],
    min=1, max=len(scene_list), step=1,
    description="Сцени:",
    continuous_update=False,
    layout=widgets.Layout(width='500px')
)
add_button = widgets.Button(description="Об'єднати вибрані сцени", button_style='info')
reset_button = widgets.Button(description="Скинути об'єднання", button_style='warning')
out = widgets.Output()

def show_merge_state():
    clear_output()
    if merge_ranges:
        print("Поточні групи об'єднання (номери сцен, включно):")
        for r in sorted(merge_ranges):
            print(f"  {r[0]}..{r[1]}")
    else:
        print("Групи об'єднання відсутні — сцени підуть як є.")

def on_add_clicked(b):
    with out:
        a, c = range_slider.value
        merge_ranges.append((a, c))
        show_merge_state()

def on_reset_clicked(b):
    with out:
        merge_ranges.clear()
        show_merge_state()

add_button.on_click(on_add_clicked)
reset_button.on_click(on_reset_clicked)

print("\nЯкщо детектор помилково розрізав одну сцену на дві — вибери повзунком")
print("діапазон номерів сцен і натисни 'Об'єднати вибрані сцени'. Можна додати")
print("декілька груп об'єднання підряд, кожна з окремим діапазоном повзунка.\n")

display(widgets.HBox([range_slider, add_button, reset_button]), out)
with out:
    show_merge_state()


In [ ]:
#@title ##**Run PySceneDetect & Save Archive** { display-mode: "form" }
import os
import time
import zipfile
import subprocess
from tqdm.notebook import tqdm

def log_time(start, message):
    elapsed = time.time() - start
    print(f"{message}: {elapsed:.2f} seconds")
    return time.time()

start_time = time.time()

# Застосовуємо ручні об'єднання сцен (якщо додавали через повзунок вище)
if merge_ranges:
    sorted_ranges = sorted(merge_ranges, key=lambda r: r[0])
    final_scene_list = []
    skip_until = 0
    for i, (start, end) in enumerate(scene_list, start=1):
        if i <= skip_until:
            continue
        group = next(((a, c) for a, c in sorted_ranges if a == i), None)
        if group:
            a, c = group
            merged_start = scene_list[a - 1][0]
            merged_end = scene_list[c - 1][1]
            final_scene_list.append((merged_start, merged_end))
            skip_until = c
        else:
            final_scene_list.append((start, end))
    scene_list = final_scene_list
    print(f"Об'єднання застосовано, після об'єднання: {len(scene_list)} сцен")
else:
    print(f"Об'єднань немає, використовуємо {len(scene_list)} сцен як є")

start_time = log_time(start_time, "Applying merges")

base_name = os.path.splitext(os.path.basename(file_name))[0]
scenes_dir = f"/content/{base_name}_scenes"
os.makedirs(scenes_dir, exist_ok=True)

# Кожну сцену виріжемо з ПЕРЕкодуванням у lossless (-crf 0), а не -c copy.
# -ss/-to ПЕРЕД -i дають точний покадровий (frame-accurate) розріз при декодуванні,
# а -crf 0 гарантує, що якість картинки не погіршиться відносно оригіналу.
scene_paths = []
for i, (start, end) in enumerate(tqdm(scene_list, desc="Splitting scenes"), start=1):
    start_tc = start.get_timecode()
    end_tc = end.get_timecode()
    out_path = os.path.join(scenes_dir, f"{base_name}-Scene-{i:03d}.mp4")

    cmd = [
        'ffmpeg', '-y',
        '-ss', start_tc, '-to', end_tc,
        '-i', file_name,
        '-c:v', 'libx264', '-preset', 'medium', '-crf', '0',
        '-c:a', 'aac', '-b:a', '192k',
        out_path
    ]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed on scene {i}: {result.stderr[-1500:]}")
    scene_paths.append(out_path)

start_time = log_time(start_time, "Scene splitting")

if output_folder == "google_drive":
    from google.colab import drive
    drive.mount('/content/drive')
    save_path = '/content/drive/MyDrive/'
else:
    save_path = '/content/'

archive_name = f"{base_name}_scenes.zip"
archive_path = os.path.join(save_path, archive_name)

with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_STORED) as zf:
    for p in tqdm(sorted(scene_paths), desc="Archiving"):
        zf.write(p, arcname=os.path.basename(p))

start_time = log_time(start_time, "Archiving")

print(f"\nArchive saved: {archive_path}")
print(f"Scenes: {len(scene_paths)}, folder used for verification: {scenes_dir}")


In [ ]:
#@title ##**Verify — Reconstruct & Compare with Original** { display-mode: "form" }
import os
import subprocess

concat_list_path = "/content/concat_list.txt"
scene_files = sorted(
    os.path.join(scenes_dir, f) for f in os.listdir(scenes_dir) if f.lower().endswith('.mp4')
)

if not scene_files:
    raise FileNotFoundError(f"No scene files found in {scenes_dir}. Run the previous cell first.")

with open(concat_list_path, 'w') as f:
    for sf in scene_files:
        f.write(f"file '{sf}'\n")

reconstructed_path = "/content/reconstructed.mp4"

# Спершу пробуємо швидке склеювання без перекодування (усі сцени вже в однаковому
# кодеку/параметрах, тож stream copy має спрацювати і не втратити якість).
cmd = ['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', concat_list_path, '-c', 'copy', reconstructed_path]
result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

if result.returncode != 0:
    print("Stream-copy склеювання не вдалося, перекодовую (lossless)...")
    cmd = ['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', concat_list_path,
           '-c:v', 'libx264', '-crf', '0', '-preset', 'medium', '-c:a', 'aac', reconstructed_path]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Concat failed: {result.stderr[-1500:]}")

def get_duration(path):
    cmd = ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
           '-of', 'default=noprint_wrappers=1:nokey=1', path]
    out = subprocess.run(cmd, stdout=subprocess.PIPE, text=True).stdout.strip()
    return float(out) if out else 0.0

def get_frame_count(path):
    cmd = ['ffprobe', '-v', 'error', '-count_frames', '-select_streams', 'v:0',
           '-show_entries', 'stream=nb_read_frames', '-of', 'default=noprint_wrappers=1:nokey=1', path]
    out = subprocess.run(cmd, stdout=subprocess.PIPE, text=True).stdout.strip()
    return int(out) if out else 0

orig_duration = get_duration(file_name)
recon_duration = get_duration(reconstructed_path)
orig_frames = get_frame_count(file_name)
recon_frames = get_frame_count(reconstructed_path)

print("=== Duration & frame count ===")
print(f"Original duration:      {orig_duration:.3f} s")
print(f"Reconstructed duration: {recon_duration:.3f} s   (Δ {abs(orig_duration - recon_duration):.3f} s)")
print(f"Original frames:        {orig_frames}")
print(f"Reconstructed frames:   {recon_frames}   (Δ {abs(orig_frames - recon_frames)})")

print("\n=== Quality (SSIM / PSNR vs original) ===")
cmd = ['ffmpeg', '-y', '-i', reconstructed_path, '-i', file_name,
       '-lavfi', '[0:v][1:v]ssim;[0:v][1:v]psnr', '-f', 'null', '-']
result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

if result.returncode == 0:
    for line in result.stderr.splitlines():
        if "SSIM" in line or "PSNR" in line:
            print(line.strip())
else:
    print("Не вдалося порахувати SSIM/PSNR (можливо, відрізняється тривалість/роздільна здатність).")
    print(result.stderr[-600:])

print("\nВисновок: якщо тривалість і кількість кадрів співпадають (або майже),")
print("а SSIM близький до 1.0 (PSNR — високий, напр. >40dB або inf), реконструкція")
print("кадр-в-кадр успішна і якість не гірша за оригінал.")
